# M&A Target Identification Engine

**Dataset:** `specific_fields_dataset.csv` — 1,433 companies × 5 sectors × 2016–2026  
**Methodology:** 5-Dimensional Scoring (Size, Profitability, Growth, Health, Public Float)  
**Output:** Ranked acquisition targets + sector summary + visualizations

---
**Weight Configuration:**
- Size: 25% | Profitability: 20% | Growth: 20% | Financial Health: 20% | Public Float: 15%

**Pipeline:** Load → Feature Engineering → Latest-Year Snapshot → Score → Rank → Visualize

In [1]:
# === CELL 1: IMPORTS & CONFIG ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from math import pi

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 120

ROOT = Path.cwd().resolve()
DATA_FILE = ROOT / "data" / "processed" / "specific_fields_dataset.csv"
OUT_RANKED = ROOT / "data" / "processed" / "ma_targets_ranked.csv"
OUT_SECTOR = ROOT / "data" / "processed" / "ma_sector_summary.csv"

print(f"Root: {ROOT}")
print(f"Dataset: {DATA_FILE}")
print(f"Exists: {DATA_FILE.exists()}")
print(f"\nLibraries loaded ✓")

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# === CELL 2: LOAD DATASET ===
df = pd.read_csv(DATA_FILE, encoding="utf-8-sig")

print(f"Rows:       {len(df):>8,}")
print(f"Columns:    {len(df.columns):>8}")
print(f"Companies:  {df['ticker'].nunique():>8,}")
print(f"Sectors:    {df['target_sector'].nunique():>8}")
print(f"Year range: {df['fiscal_year'].min()} – {df['fiscal_year'].max()}")
print()
print("Sectors:")
for s in sorted(df['target_sector'].unique()):
    n = df[df['target_sector'] == s]['ticker'].nunique()
    print(f"  • {s}: {n} companies")

df.head()

---
## Feature Engineering

We compute the following derived fields from the raw financial data:
- **net_margin** = net_income / total_revenue
- **debt_to_equity** = total_liabilities / (total_assets − total_liabilities) → clamped to [0, 20]
- **current_ratio** = current_assets / current_liabilities
- **revenue_growth** = 1-year YoY % change → clamped to [−90%, +200%]
- **revenue_growth_3yr** = 3-year average annualized growth → clamped to [−90%, +200%]
- **equity** = total_assets − total_liabilities

Growth rates are clamped to prevent one-off events (e.g. a tiny company doubling revenue on one contract) from distorting scores.

In [ ]:
# === CELL 3: FEATURE ENGINEERING ===
df = df.sort_values(["ticker", "fiscal_year"]).copy()

# Net margin
df["net_margin"] = np.divide(
    df["net_income"], df["total_revenue"],
    out=np.full(len(df), np.nan),
    where=df["total_revenue"].fillna(0).ne(0),
)

# Debt-to-equity
equity = df["total_assets"] - df["total_liabilities"]
df["debt_to_equity"] = np.divide(
    df["total_liabilities"], equity,
    out=np.full(len(df), np.nan),
    where=equity.fillna(0).ne(0),
).clip(0, 20)

# Current ratio
df["current_ratio"] = np.divide(
    df["current_assets"], df["current_liabilities"],
    out=np.full(len(df), np.nan),
    where=df["current_liabilities"].fillna(0).ne(0),
)

# Revenue growth (1-year YoY, clamped)
df["revenue_growth"] = df.groupby("ticker")["total_revenue"].pct_change(1)
df["revenue_growth"] = df["revenue_growth"].clip(-0.90, 2.0)

# Revenue growth (3-year average annualized, clamped)
df["revenue_growth_3yr"] = (
    df.groupby("ticker")["total_revenue"]
    .transform(lambda x: x.pct_change(3) / 3)
    .clip(-0.90, 2.0)
)

df["equity"] = equity

print("Derived fields created:")
for col in ["net_margin", "debt_to_equity", "current_ratio", "revenue_growth", "revenue_growth_3yr", "equity"]:
    coverage = df[col].notna().mean() * 100
    print(f"  {col:25s}  {coverage:5.1f}% coverage  ("
          f"median: {df[col].median():.3f}, "
          f"99th: {df[col].quantile(0.99):.3f})")

df[['ticker', 'fiscal_year', 'total_revenue', 'revenue_growth', 'revenue_growth_3yr', 'net_margin']].head(8)

---
## Latest-Year Snapshot

We take the **most recent fiscal year** for each company. This gives us ~1,400 companies to score. Companies with zero financial data (no revenue, no assets) are dropped.

In [ ]:
# === CELL 4: LATEST YEAR SNAPSHOT ===
latest = (
    df.sort_values("fiscal_year")
    .groupby("ticker", as_index=False)
    .last()
    .copy()
)

# Drop companies without any financial data
before = len(latest)
latest = latest[latest[["total_assets", "total_revenue"]].notna().any(axis=1)].copy()
dropped = before - len(latest)

print(f"Companies before filter: {before:,}")
print(f"Dropped (no data):      {dropped:,}")
print(f"Companies to score:     {len(latest):,}")
print(f"\nFiscal year distribution:")
print(latest["fiscal_year"].value_counts().sort_index().to_string())

---
## Scoring Functions — 5 Dimensions

Each dimension is scored 0–100 using **sector-relative percentile ranks** (a company is compared only to peers in its own sector):

| # | Dimension | Weight | Method |
|---|-----------|:------:|--------|
| 1 | **Size** | 25% | Inverse percentile rank of total_assets. Bonus for < $50M assets. |
| 2 | **Profitability** | 20% | Percentile rank of net margin + flag if net_income > 0. |
| 3 | **Growth** | 20% | Blend of 1yr (60%) and 3yr (40%) revenue growth percentile ranks. |
| 4 | **Financial Health** | 20% | Inverse leverage rank + current ratio rank + earnings quality flags. |
| 5 | **Public Float** | 15% | Inverse percentile rank of market_value. Bonus for < $500M. |

The **composite score** = weighted average of all 5 dimensions.

In [ ]:
# === CELL 5: SCORING FUNCTIONS ===

def percentile_rank(s):
    """Return percentile rank 0-1 (higher = higher rank)."""
    return s.rank(pct=True)

def inverse_percentile_rank(s):
    """Return inverse percentile rank 0-1 (higher = smaller)."""
    return 1 - s.rank(pct=True)


def score_size(data):
    """
    Size Score (25% weight) — smaller companies are easier to acquire.
    Uses total_assets; bonus for micro-cap (< $50M).
    """
    raw = data.groupby("target_sector")["total_assets"].transform(
        lambda x: inverse_percentile_rank(x.fillna(x.median()))
    ).fillna(0.5)
    bonus = (data["total_assets"].fillna(0) < 50_000_000).astype(float) * 0.15
    return (raw + bonus).clip(0, 1) * 100


def score_profit(data):
    """
    Profitability Score (20% weight) — profitable companies = accretive acquisitions.
    Blend of margin percentile (60%) + profitability flag (40%).
    """
    margin = data.groupby("target_sector")["net_margin"].transform(
        lambda x: percentile_rank(x.fillna(0))
    ).fillna(0.5)
    profit_flag = (data["net_income"].fillna(0) > 0).astype(float)
    return (margin * 0.60 + profit_flag * 0.40) * 100


def score_growth(data):
    """
    Growth Score (20% weight) — growing revenue = future upside.
    Blend of 1yr (60%) and 3yr (40%) growth ranks.
    """
    g1 = data.groupby("target_sector")["revenue_growth"].transform(
        lambda x: percentile_rank(x.fillna(0))
    ).fillna(0.5)
    g3 = data.groupby("target_sector")["revenue_growth_3yr"].transform(
        lambda x: percentile_rank(x.fillna(0))
    ).fillna(0.5)
    return (g1 * 0.60 + g3 * 0.40) * 100


def score_health(data):
    """
    Financial Health Score (20% weight) — healthy targets close faster.
    Combines low leverage, good liquidity, and earnings quality.
    """
    dte = data.groupby("target_sector")["debt_to_equity"].transform(
        lambda x: inverse_percentile_rank(x.fillna(x.median()))
    ).fillna(0.5)
    cr = data.groupby("target_sector")["current_ratio"].transform(
        lambda x: percentile_rank(x.fillna(1.0))
    ).fillna(0.5)
    ni = (data["net_income"].fillna(0) > 0).astype(float)
    re = (data["retained_earnings"].fillna(0) > 0).astype(float)
    return (dte * 0.30 + cr * 0.30 + ni * 0.20 + re * 0.20) * 100


def score_float(data):
    """
    Public Float Score (15% weight) — smaller float = fewer shareholders to convince.
    Uses market_value (EntityPublicFloat).
    """
    raw = data.groupby("target_sector")["market_value"].transform(
        lambda x: inverse_percentile_rank(x.fillna(x.median()))
    ).fillna(0.5)
    bonus = (data["market_value"].fillna(0) < 500_000_000).astype(float) * 0.10
    return (raw + bonus).clip(0, 1) * 100


print("All 5 scoring functions defined ✓")
print("\nScores are sector-relative percentiles (0–100)")

In [ ]:
# === CELL 6: APPLY SCORES, COMPOSITE, THESIS LABELS, RANK ===

# --- 6a: Apply all 5 scoring functions ---
latest["size_score"]    = score_size(latest)
latest["profit_score"]  = score_profit(latest)
latest["growth_score"]  = score_growth(latest)
latest["health_score"]  = score_health(latest)
latest["mv_score"]      = score_float(latest)

# --- 6b: Composite weighted score ---
WEIGHTS = {
    "size": 0.25,
    "profitability": 0.20,
    "growth": 0.20,
    "financial_health": 0.20,
    "public_float": 0.15,
}

latest["acquirability_score"] = (
    latest["size_score"]   * WEIGHTS["size"] +
    latest["profit_score"] * WEIGHTS["profitability"] +
    latest["growth_score"] * WEIGHTS["growth"] +
    latest["health_score"] * WEIGHTS["financial_health"] +
    latest["mv_score"]     * WEIGHTS["public_float"]
)

# --- 6c: Generate investment thesis labels ---
def thesis(row):
    tags = []
    if row["size_score"] >= 80:
        tags.append("Micro-cap")
    elif row["size_score"] >= 60:
        tags.append("Small-cap")
    elif row["size_score"] <= 30:
        tags.append("Large-cap")
    else:
        tags.append("Mid-cap")
    
    if row["growth_score"] >= 75:
        gr = row.get("revenue_growth", 0) or 0
        tags.append("high-growth" if gr > 0.3 else "steady-grower")
    
    if row["profit_score"] >= 75:
        tags.append("high-margin")
    elif row["profit_score"] <= 35:
        tags.append("turnaround-candidate")
    
    if row["health_score"] >= 75:
        tags.append("low-debt")
    elif row["health_score"] <= 35:
        tags.append("distressed")
    
    return f"{' / '.join(tags)} {row['target_sector']} target"

latest["thesis"] = latest.apply(thesis, axis=1)

# --- 6d: Rank ---
latest = latest.sort_values("acquirability_score", ascending=False).reset_index(drop=True)
latest["rank"] = range(1, len(latest) + 1)

# --- Summary stats ---
print("SCORES SUMMARY")
print(f"{'Score':25s} {'Mean':>7s} {'Median':>7s} {'Std':>7s} {'Min':>7s} {'Max':>7s}")
print("-" * 60)
for col in ["acquirability_score", "size_score", "profit_score",
            "growth_score", "health_score", "mv_score"]:
    s = latest[col].dropna()
    print(f"{col:25s} {s.mean():>7.1f} {s.median():>7.1f} {s.std():>7.1f} {s.min():>7.1f} {s.max():>7.1f}")

---
## Results — Top 50 Acquisition Targets

Each company is shown with:
- **Score:** Composite acquirability (0–100)
- **Revenue:** Latest annual revenue
- **Growth:** YoY revenue growth rate
- **Thesis:** One-line explanation of *why* it's a target

In [ ]:
# === CELL 7: DISPLAY TOP 50 ===
display_cols = [
    "rank", "ticker", "company", "target_sector",
    "fiscal_year", "total_revenue", "net_income",
    "acquirability_score", "size_score", "profit_score",
    "growth_score", "health_score", "mv_score",
    "revenue_growth", "net_margin", "thesis",
]
display_cols = [c for c in display_cols if c in latest.columns]

top50 = latest.head(50)[display_cols].copy()

def fmt_rev(x):
    if pd.isna(x) or x == 0:
        return "N/A"
    if abs(x) >= 1_000_000_000:
        return f"${x/1_000_000_000:.1f}B"
    if abs(x) >= 1_000_000:
        return f"${x/1_000_000:.0f}M"
    return f"${x:,.0f}"

print("=" * 100)
print("TOP 50 ACQUISITION TARGETS — RANKED BY ACQUIRABILITY SCORE")
print("=" * 100)
print()

for _, row in top50.iterrows():
    rev = fmt_rev(row.get("total_revenue", 0)) if pd.notna(row.get("total_revenue")) else "N/A"
    growth = f"{row['revenue_growth']*100:.0f}%" if pd.notna(row.get("revenue_growth")) else "N/A"
    score = round(row["acquirability_score"], 1)
    thesis_str = row.get("thesis", "")
    
    print(f"  {int(row['rank']):>3}. {row['ticker']:<8s} | "
          f"{str(row.get('target_sector',''))[:14]:14s} | "
          f"Score: {score:5.1f} | "
          f"Rev: {rev:>8s} | "
          f"Growth: {growth:>5s} | "
          f"{thesis_str}")

print()
print(f"Score distribution: {latest['acquirability_score'].min():.1f} – "
      f"{latest['acquirability_score'].max():.1f} "
      f"(median: {latest['acquirability_score'].median():.1f})")

---
## Visualizations

Four charts to understand the results:
1. **Sector Score Breakdown** — Bar chart: 5 sectors × 5 dimensions
2. **Score Distribution** — Histograms: composite + each dimension
3. **Radar Chart** — Profile comparison of the top 3 targets
4. **Sector Heatmap** — Color-coded dimension scores per sector

In [ ]:
# === CELL 8: SECTOR SCORE BREAKDOWN (BAR CHART) ===
sector_scores = latest.groupby("target_sector").agg(
    avg_score=("acquirability_score", "mean"),
    size=("size_score", "mean"),
    profit=("profit_score", "mean"),
    growth=("growth_score", "mean"),
    health=("health_score", "mean"),
    mv=("mv_score", "mean"),
    top_target=("ticker", "first"),
    count=("ticker", "count"),
).round(1)

fig, ax = plt.subplots(figsize=(12, 5))
sector_scores[["size", "profit", "growth", "health", "mv"]].plot(
    kind="bar", ax=ax, colormap="Set2", edgecolor="gray", width=0.8
)
ax.set_title("M&A Target Score Breakdown by Sector", fontsize=14, fontweight="bold")
ax.set_ylabel("Average Score (0–100)")
ax.set_xlabel("")
ax.legend(title="Dimension", bbox_to_anchor=(1.0, 1))
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right")

# Add composite score label on top of each sector
for i, (sector, row) in enumerate(sector_scores.iterrows()):
    ax.text(i, row[["size","profit","growth","health","mv"]].max() + 2,
            f"Score: {row['avg_score']:.1f}", ha="center", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()

print("\nSector Summary:")
print("=" * 50)
print(f"{'Sector':22s} {'Count':>6s} {'Score':>7s} {'Top Target':>12s}")
print("-" * 50)
for sector, row in sector_scores.iterrows():
    print(f"{sector:22s} {int(row['count']):>6,d} {row['avg_score']:>6.1f}  {row['top_target']:>12s}")

In [ ]:
# === CELL 9: SCORE DISTRIBUTION (HISTOGRAM) ===
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

score_cols = ["acquirability_score", "size_score", "profit_score",
              "growth_score", "health_score", "mv_score"]
titles = ["Composite Score", "Size", "Profitability",
          "Growth", "Financial Health", "Public Float"]

for ax, col, t in zip(axes, score_cols, titles):
    vals = latest[col].dropna()
    ax.hist(vals, bins=30, color="steelblue", edgecolor="white", alpha=0.8)
    ax.axvline(vals.mean(), color="red", linestyle="--",
               label=f"Mean: {vals.mean():.1f}")
    ax.axvline(vals.median(), color="green", linestyle=":",
               label=f"Median: {vals.median():.1f}")
    ax.set_title(t, fontweight="bold")
    ax.set_xlabel("Score (0–100)")
    ax.set_ylabel("Companies")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# === CELL 10: RADAR CHART — TOP 3 TARGETS ===
top3 = latest.head(3).copy()
dimensions = ["size_score", "profit_score", "growth_score", "health_score", "mv_score"]
dim_labels = ["Size", "Profit", "Growth", "Health", "Float"]
N = len(dimensions)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

colors = ["#e41a1c", "#377eb8", "#4daf4a"]
for i, (_, row) in enumerate(top3.iterrows()):
    values = row[dimensions].values.tolist()
    values += values[:1]
    ax.fill(angles, values, alpha=0.1, color=colors[i])
    ax.plot(angles, values, color=colors[i], linewidth=2,
            label=f"#{int(row['rank'])} {row['ticker']} ({row['acquirability_score']:.1f})")

ax.set_xticks(angles[:-1])
ax.set_xticklabels(dim_labels, fontsize=12)
ax.set_ylim(0, 100)
ax.set_title("Top 3 M&A Targets — Dimension Profile Comparison",
             fontsize=14, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()

# Print thesis for each
for _, row in top3.iterrows():
    print(f"#{int(row['rank'])} {row['ticker']} — {row['thesis']}")

In [ ]:
# === CELL 11: HEATMAP + EXPORT ===
fig, ax = plt.subplots(figsize=(10, 5))
heatmap_data = sector_scores[["size", "profit", "growth", "health", "mv"]]

sns.heatmap(heatmap_data, annot=True, fmt=".1f", cmap="YlOrRd",
            linewidths=0.5, ax=ax,
            cbar_kws={"label": "Average Score (0–100)"})
ax.set_title("M&A Target Dimension Scores by Sector", fontweight="bold", fontsize=13)
ax.set_ylabel("")
ax.set_xlabel("Scoring Dimension")
plt.tight_layout()
plt.show()

# --- Export to CSV ---
export_cols = [c for c in display_cols if c in latest.columns]
OUT_RANKED.parent.mkdir(parents=True, exist_ok=True)
latest[export_cols].to_csv(OUT_RANKED, index=False, encoding="utf-8-sig")
sector_sums = sector_scores[["avg_score", "size", "profit", "growth", "health", "mv", "count"]]
sector_sums.columns = ["avg_score", "size", "profit", "growth", "health", "float", "companies"]
sector_sums.to_csv(OUT_SECTOR, encoding="utf-8-sig")

print(f"\n{'='*50}")
print("EXPORT COMPLETE")
print(f"{'='*50}")
print(f"  Ranked targets: {OUT_RANKED.name} ({len(latest):,} companies)")
print(f"  Sector summary: {OUT_SECTOR.name} ({len(sector_sums)} sectors)")
print()
print(f"Top overall target: #{1} {latest.iloc[0]['ticker']} ({latest.iloc[0]['company']})")
print(f"  Score: {latest.iloc[0]['acquirability_score']:.1f}/100")
print(f"  Thesis: {latest.iloc[0]['thesis']}")
print()
print("DONE ✓")

---
## Next Steps & Analysis Ideas

### Weight Tuning
Try different weight profiles to match specific investment strategies.
The reference `RESEARCH/ma_target_identification.md` has 5 presets:
| Profile | Size | Profit | Growth | Health | Float |
|---------|:----:|:------:|:------:|:------:|:-----:|
| Balanced (default) | 0.25 | 0.20 | 0.20 | 0.20 | 0.15 |
| PE Buyout | 0.30 | 0.25 | 0.10 | 0.25 | 0.10 |
| Strategic Growth | 0.15 | 0.15 | 0.35 | 0.20 | 0.15 |
| Distressed / Turnaround | 0.20 | 0.05 | 0.10 | 0.05 | 0.60 |
| Micro-Cap Focus | 0.35 | 0.20 | 0.20 | 0.15 | 0.10 |

### Deeper Analysis
1. **3-year trend overlay** — How many years has a target been growing? Stable improves thesis.
2. **Sector-specific screening** — Filter to one sector and re-rank within it.
3. **Revenue size filter** — Add a minimum revenue threshold (e.g. > $10M for institutional buyers).
4. **Cross-reference with actual M&A** — Compare targets to real deal data (if available).
5. **Hypothetical portfolio** — Pick the top 5 targets per sector and track them over time.

### Limitations
- No ownership structure data (dual-class shares, founder control)
- No deal premium / valuation data (scores fundamental attractiveness only)
- Broad sector classification ("Industrials" covers many sub-industries)
- Snapshot is one year only (latest available fiscal year)